In [ ]:
# This will install the newest wormtracer into colab environments.
# After installation, you should restart the jupyter session
%%bash

pip install "wormtracer @ git+https://github.com/lycantrope/WormTracer.git"

In [ ]:
# Restart kernel after pip install
from google.colab import runtime
runtime.unassign() 


In [ ]:
%%writefile params.yaml
# This cell uses %%writefile to create a `params.yaml` file
# This file contains configuration parameters for the WormTracer application.
# Users can modify these values to adjust the tracing behavior.

# Local time difference parameter
local_time_difference: 9

# Start and end frames for processing (0 means process all frames)
start_T: 0
end_T: 0

# Rescaling factors for image and time
rescale: 1
Tscale: 1

# Weights for different loss functions used during optimization
continuity_loss_weight: 10000
smoothness_loss_weight: 100000
length_loss_weight: 50
center_loss_weight: 50

# Plotting and epoch parameters
plot_n: 100
epoch_plus: 1500
speed: 0.05
lr: 0.05
body_ratio: 90
judge_head_method: frequency

# Display and saving options for progress and results
num_t: 5
ShowProgress: False
SaveProgress: False
show_progress_freq: 200
save_progress_freq: 50
save_progress_num: 50

SaveCenterlinedWormsSerial: False
SaveCenterlinedWormsMovie: False
SaveCenterlinedWormsMultitiff: False

In [ ]:
%%bash
# This cell downloads sample data required by WormTracer.
# It creates a 'test' directory, navigates into it, and downloads a zip file.
# Finally, it unzips the downloaded file containing the WT_binary.tif image.
# You can replace this part with code for mounting Google Drive or other data sources.
mkdir test
cd test
wget https://github.com/lycantrope/WormTracer/raw/refs/heads/main/Sample%20Images/WT_binary_multipage_tiff.zip
unzip WT_binary_multipage_tiff.zip

In [ ]:
from WormTracer import wt

# Run regular wormtracer without any options.
wt.run(
    parameter_file="/content/params.yaml",
    dataset_path="/content/test",
)


In [ ]:
%%bash
# Create Zip file
zip -r "wt_test_res.zip" "./test/test_output_001"

In [ ]:
# Download the results
from google.colab import files
files.download("wt_test_res.zip")

## Re-run wormtracer with guide files

In [ ]:
from WormTracer import wt

# To use guide files, you must assign x and y coordinates by assigning the name containing _x and _y, respectively.
guidecsv_x_path = "/path/to/your/guide/file/xxxx_guide_x.csv"
guidecsv_y_path = "/path/to/your/guide/file/xxxx_guide_y.csv"

# 
# Running with CSV guide files.
wt.run(
    parameter_file="/content/params.yaml",
    dataset_path="/content/test",
    guide_files= [guidecsv_x_path, guidecsv_y_path],
)



In [ ]:
import h5py

# If the input file is HDF5. Then the HDF must contains two datasets named as x and y with same shape 
guide_hdf = "/path/to/your/guide/file/xxxx_guide.h5"

# Following code is only for Sanity test, can be omitted  
with h5py.File(guide_hdf, "r") as fd:
    assert set(fd.keys()) == set(("x", "y")), "Guide HDF must be `x` and `y`"

# Running with HDF guide files.
wt.run(
    parameter_file="/content/params.yaml",
    dataset_path="/content/test",
    guide_files= [guide_hdf],
)